# Day 19 — Sorting and top-k

*100 Days of Polars · Phase 2: Expressions*

- **Post:** {{TODO: Medium link}}
- **Dataset:** [Olist Brazilian e-commerce](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) — CSVs in `data/raw/`
- **Written for:** Polars 2.x (run from the repo root)

In [26]:
import polars as pl

print("polars", pl.__version__)

polars 2.0.0


Sorting a whole frame, sorting **inside** a group, and getting the top k without sorting everything are three different tools.

In [27]:
import time

items = pl.read_csv("data/raw/olist_order_items_dataset.csv", try_parse_dates=True)
orders = pl.read_csv("data/raw/olist_orders_dataset.csv", try_parse_dates=True)

## 1. `DataFrame.sort`

In [28]:
items.sort("price", descending=True).head(5)

order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
str,i64,str,str,datetime[μs],f64,f64
"""0812eb902a67711a1cb742b3cdaa65…",1,"""489ae2aa008f021502940f251d4cce…","""e3b4998c7a498169dc7bce44e6bb62…",2017-02-16 20:37:36,6735.0,194.31
"""fefacc66af859508bf1a7934eab1e9…",1,"""69c590f7ffc7bf8db97190b6cb6ed6…","""80ceebb4ee9b31afb6c6a916a574a1…",2018-08-02 04:05:13,6729.0,193.21
"""f5136e38d1a14a4dbd87dff67da827…",1,"""1bdf5e6731585cf01aa8169c7028d6…","""ee27a8f15b1dded4d213a468ba4eb3…",2017-06-15 02:45:17,6499.0,227.66
"""a96610ab360d42a2e5335a3998b471…",1,"""a6492cc69376c469ab6f61d8f44de9…","""59417c56835dd8e2e72f91f809cd40…",2017-04-18 13:25:18,4799.0,151.34
"""199af31afc78c699f0dbf71fb178d4…",1,"""c3ed642d592594bb648ff4a04cee27…","""59417c56835dd8e2e72f91f809cd40…",2017-05-09 15:50:15,4690.0,74.34


In [29]:
# Several keys, each with its own direction
items.sort(["seller_id", "price"], descending=[False, True]).head(5)

order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
str,i64,str,str,datetime[μs],f64,f64
"""9dc8d1a6f16f1b89874c29c9d8d304…",1,"""a2ff5a97bf95719e38ea2e3b4105bc…","""0015a82c2db000af6aaaf3ae2ecb05…",2017-10-18 14:49:22,895.0,21.02
"""7f39ba4c9052be115350065d07583c…",1,"""a2ff5a97bf95719e38ea2e3b4105bc…","""0015a82c2db000af6aaaf3ae2ecb05…",2017-10-24 23:56:20,895.0,21.02
"""d455a8cb295653b55abda06d434ab4…",1,"""a2ff5a97bf95719e38ea2e3b4105bc…","""0015a82c2db000af6aaaf3ae2ecb05…",2017-10-12 22:24:16,895.0,21.02
"""14bba6d3df6ead296d2626315da5e5…",1,"""e251ebd2858be1aa7d9b2087a69925…","""001cca7ae9ae17fb1caed9dfb10948…",2017-05-05 15:10:21,199.0,17.09
"""67228d3b35b57e5bf9c1437f95652f…",1,"""98a8c2fa16d7239c606640f5555768…","""001cca7ae9ae17fb1caed9dfb10948…",2018-04-17 11:10:23,169.0,114.62


In [30]:
# Sort by an expression without adding a column
items.sort(pl.col("price") + pl.col("freight_value"), descending=True).select("order_id", "price", "freight_value").head(5)

order_id,price,freight_value
str,f64,f64
"""0812eb902a67711a1cb742b3cdaa65…",6735.0,194.31
"""fefacc66af859508bf1a7934eab1e9…",6729.0,193.21
"""f5136e38d1a14a4dbd87dff67da827…",6499.0,227.66
"""a96610ab360d42a2e5335a3998b471…",4799.0,151.34
"""199af31afc78c699f0dbf71fb178d4…",4690.0,74.34


In [31]:
# Where do nulls go? First by default
print(orders.sort("order_delivered_customer_date")["order_delivered_customer_date"].head(3).to_list())
print(orders.sort("order_delivered_customer_date", nulls_last=True)["order_delivered_customer_date"].head(3).to_list())

[None, None, None]
[datetime.datetime(2016, 10, 11, 13, 46, 32), datetime.datetime(2016, 10, 11, 14, 46, 49), datetime.datetime(2016, 10, 13, 3, 10, 34)]


## 2. Gotcha: `Expr.sort` sorts one column on its own

Inside `select`, `pl.col("price").sort()` sorts **that column only**. The other columns stay where they were, so rows no longer line up.

In [32]:
items.select("order_id", pl.col("price").sort(descending=True)).head(3)  # prices no longer belong to these orders!

order_id,price
str,f64
"""00010242fe8c5a6d1ba2dd792cb162…",6735.0
"""00018f77f2f0320c557190d7a144bd…",6729.0
"""000229ec398224ef6ca0657da4fc70…",6499.0


In [33]:
# Keep rows together: give every column the same order
print(items.sort("price", descending=True).select("order_id", "price").head(3))   # sort the frame
print(items.select(pl.all().sort_by("price", descending=True)).select("order_id", "price").head(3))  # same key for all columns

shape: (3, 2)
┌─────────────────────────────────┬────────┐
│ order_id                        ┆ price  │
│ ---                             ┆ ---    │
│ str                             ┆ f64    │
╞═════════════════════════════════╪════════╡
│ 0812eb902a67711a1cb742b3cdaa65… ┆ 6735.0 │
│ fefacc66af859508bf1a7934eab1e9… ┆ 6729.0 │
│ f5136e38d1a14a4dbd87dff67da827… ┆ 6499.0 │
└─────────────────────────────────┴────────┘
shape: (3, 2)
┌─────────────────────────────────┬────────┐
│ order_id                        ┆ price  │
│ ---                             ┆ ---    │
│ str                             ┆ f64    │
╞═════════════════════════════════╪════════╡
│ 0812eb902a67711a1cb742b3cdaa65… ┆ 6735.0 │
│ fefacc66af859508bf1a7934eab1e9… ┆ 6729.0 │
│ f5136e38d1a14a4dbd87dff67da827… ┆ 6499.0 │
└─────────────────────────────────┴────────┘


In [34]:
# Where Expr.sort is safe: when that one column is the whole answer
print(items.select(pl.col("price").sort(descending=True)).head(3))                       # alone: nothing to misalign
items.group_by("order_id").agg(prices=pl.col("price").sort(descending=True)).head(3)      # inside agg: per group

shape: (3, 1)
┌────────┐
│ price  │
│ ---    │
│ f64    │
╞════════╡
│ 6735.0 │
│ 6729.0 │
│ 6499.0 │
└────────┘


order_id,prices
str,list[f64]
"""56d1f75d4f625ce5ac6ab30d22cc35…",[21.99]
"""f0ffa7b0eb3d20f79c534119564717…",[189.9]
"""8336334036e8b201a6013cebe4ef9d…",[35.99]


To sort rows, sort the frame. Inside an aggregation, use `sort_by` to order one column by another.

## 3. `sort_by` inside `agg`

In [35]:
# WRONG: sort() orders product_id by its OWN values, so first() is the alphabetically last id
(
    items.group_by("order_id")
    .agg(
        n_items=pl.len(),
        top_product=pl.col("product_id").sort(descending=True).first(),   # wrong
        top_price=pl.col("price").max(),
    )
    .filter(pl.col("n_items") > 1)
    .head(5)
)

order_id,n_items,top_product,top_price
str,u32,str,f64
"""0f9458a3e4a303f60f99659d29645e…",3,"""a19b6951c75da43aad691622dd2f6a…",38.4
"""0a7f4604ee9cab65020038275efe1d…",3,"""67ff96903c597ba4b1d7f668734bc3…",11.27
"""7a485479dbdf94502e471dbd3329a4…",2,"""fc8595227785863846b7daa2ae3f2c…",139.0
"""95329bc75845ea5b52290fba0e025e…",3,"""fafe25fbb0bcbd7b5112ab00709407…",19.99
"""3c4ec7795b9ef9b762cb33214c19c8…",2,"""810cb3c5af006624fa32d362077af8…",47.5


In [36]:
# The most expensive product in each multi-item order
(
    items.group_by("order_id")
    .agg(
        n_items=pl.len(),
        top_product=pl.col("product_id").sort_by("price", descending=True).first(),
        top_price=pl.col("price").max(),
    )
    .filter(pl.col("n_items") > 1)
    .head(5)
)

order_id,n_items,top_product,top_price
str,u32,str,f64
"""1c80798b60e75536e4c008facd7e92…",2,"""ba091f760e02c4b603a0a6a52c2d31…",135.0
"""df71a04c6a6a25fb6a6a94f30fa735…",2,"""2f13d1dc8b4e1d9d8027be50339546…",105.0
"""35f4c21fc613b36a1a2e893f424280…",2,"""e316e2909614743a28695b701bca9d…",30.0
"""589eac637e68da58add10ee5e6710b…",3,"""fe59a1e006df3ac42bf0ceb876d709…",599.0
"""5898decd34c5ff901d78522ed4722c…",2,"""49659dedf501090b249d0b09ca7faa…",49.0


In [37]:
# How often does the wrong version pick a different product than sort_by?
check = (
    items.group_by("order_id")
    .agg(
        n_items=pl.len(),
        wrong=pl.col("product_id").sort(descending=True).first(),
        right=pl.col("product_id").sort_by("price", descending=True).first(),
    )
    .filter(pl.col("n_items") > 1)
)
print(check.filter(pl.col("wrong") != pl.col("right")).height, "of", check.height, "multi-item orders get the wrong product")

1632 of 9803 multi-item orders get the wrong product


## 4. Top-k without a full sort

In [38]:
# DataFrame.top_k takes `by` as a keyword-only argument: top_k(5, by="price"), not top_k(5, "price")
items.top_k(5, by="price").sort("price", descending=True)

order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
str,i64,str,str,datetime[μs],f64,f64
"""0812eb902a67711a1cb742b3cdaa65…",1,"""489ae2aa008f021502940f251d4cce…","""e3b4998c7a498169dc7bce44e6bb62…",2017-02-16 20:37:36,6735.0,194.31
"""fefacc66af859508bf1a7934eab1e9…",1,"""69c590f7ffc7bf8db97190b6cb6ed6…","""80ceebb4ee9b31afb6c6a916a574a1…",2018-08-02 04:05:13,6729.0,193.21
"""f5136e38d1a14a4dbd87dff67da827…",1,"""1bdf5e6731585cf01aa8169c7028d6…","""ee27a8f15b1dded4d213a468ba4eb3…",2017-06-15 02:45:17,6499.0,227.66
"""a96610ab360d42a2e5335a3998b471…",1,"""a6492cc69376c469ab6f61d8f44de9…","""59417c56835dd8e2e72f91f809cd40…",2017-04-18 13:25:18,4799.0,151.34
"""199af31afc78c699f0dbf71fb178d4…",1,"""c3ed642d592594bb648ff4a04cee27…","""59417c56835dd8e2e72f91f809cd40…",2017-05-09 15:50:15,4690.0,74.34


In [39]:
items.bottom_k(5, by="price").sort("price")

order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
str,i64,str,str,datetime[μs],f64,f64
"""c5bdd8ef3c0ec420232e6683021791…",2,"""8a3254bee785a526d548a81a9bc3c9…","""96804ea39d96eb908e7c3afdb671bb…",2018-05-07 02:55:22,0.85,22.3
"""3ee6513ae7ea23bdfab5b9ab60bffc…",1,"""8a3254bee785a526d548a81a9bc3c9…","""96804ea39d96eb908e7c3afdb671bb…",2018-05-04 03:55:26,0.85,18.23
"""6e864b3f0ec71031117ad4cf46b7f2…",1,"""8a3254bee785a526d548a81a9bc3c9…","""96804ea39d96eb908e7c3afdb671bb…",2018-05-02 20:30:34,0.85,18.23
"""8272b63d03f5f79c56e9e4120aec44…",9,"""05b515fdc76e888aada3c6d66c201d…","""2709af9587499e95e803a6498a5a56…",2017-07-21 18:25:23,1.2,7.89
"""8272b63d03f5f79c56e9e4120aec44…",1,"""270516a3f41dc035aa87d220228f84…","""2709af9587499e95e803a6498a5a56…",2017-07-21 18:25:23,1.2,7.89


In [ ]:
# reverse=True on one key is the same as bottom_k
a = items.top_k(5, by="price", reverse=True).sort("price")
b = items.bottom_k(5, by="price").sort("price")
print("same rows:", a.select("price").equals(b.select("price")))

# reverse earns its place with several keys in mixed directions: highest price, then lowest freight
items.top_k(5, by=["price", "freight_value"], reverse=[False, True]).sort(["price", "freight_value"], descending=[True, False])

In [40]:
# Per group: each seller's 3 most expensive items
(
    items.group_by("seller_id")
    .agg(top3_prices=pl.col("price").top_k(3), top3_products=pl.col("product_id").top_k_by("price", k=3))
    .head(5)
)

seller_id,top3_prices,top3_products
str,list[f64],list[str]
"""a67780aba9dd436e7ac0aa1f889e73…","[171.9, 171.9, 171.9]","[""ceb8943f1e828dc9b58eeb0d0c83a377"", ""b0d49de379c3f4c50baf1b2d776f5c54"", ""29ef8f163a85d31f5a88354c0007a6d1""]"
"""aeb36d67f1414f234d42a29e27cf1f…","[765.0, 949.9, 949.9]","[""48ceb46f15a81d0d872c973bfbe3055d"", ""009df2b0bc078648fc4f5898de8cabff"", ""e337509290028e0c81c0ac93d7b1234e""]"
"""001cca7ae9ae17fb1caed9dfb10948…","[199.0, 169.0, 139.9]","[""e251ebd2858be1aa7d9b2087a6992580"", ""98a8c2fa16d7239c606640f5555768e4"", ""e251ebd2858be1aa7d9b2087a6992580""]"
"""51e4e89242cbb846c2deff3dc01565…","[119.0, 68.9, 68.9]","[""137e9ec1c850527fe02f316333848d59"", ""06efa56b8557e1efcb2c107f55d1a8dd"", ""06efa56b8557e1efcb2c107f55d1a8dd""]"
"""c4d51195486dc781531876a7d00453…","[2428.0, 1224.99, 839.99]","[""a859dc323c1bebb21c887054599fccce"", ""a578af9e0a32506a9bd6239a50cf928b"", ""6767719f80aabbbf16ab2491899c32d9""]"


`top_k` does not promise any order in its output. Sort afterwards if order matters.

## 5. Is top-k faster?

A full sort is O(n log n). Top-k only needs to keep k candidates. On 112k rows the difference is small; try it on bigger data later.

In [41]:
# top_k promises WHICH rows, not their order: compare the raw output with a sorted one
raw = items.top_k(5, by="price").select("order_id", "price")
print(raw)                                          # the right 5 rows, order not guaranteed
print(raw.sort("price", descending=True))           # sort the small result when order matters
print("first row is the max:", raw["price"][0] == items["price"].max())

shape: (5, 2)
┌─────────────────────────────────┬────────┐
│ order_id                        ┆ price  │
│ ---                             ┆ ---    │
│ str                             ┆ f64    │
╞═════════════════════════════════╪════════╡
│ 0812eb902a67711a1cb742b3cdaa65… ┆ 6735.0 │
│ fefacc66af859508bf1a7934eab1e9… ┆ 6729.0 │
│ f5136e38d1a14a4dbd87dff67da827… ┆ 6499.0 │
│ a96610ab360d42a2e5335a3998b471… ┆ 4799.0 │
│ 199af31afc78c699f0dbf71fb178d4… ┆ 4690.0 │
└─────────────────────────────────┴────────┘
shape: (5, 2)
┌─────────────────────────────────┬────────┐
│ order_id                        ┆ price  │
│ ---                             ┆ ---    │
│ str                             ┆ f64    │
╞═════════════════════════════════╪════════╡
│ 0812eb902a67711a1cb742b3cdaa65… ┆ 6735.0 │
│ fefacc66af859508bf1a7934eab1e9… ┆ 6729.0 │
│ f5136e38d1a14a4dbd87dff67da827… ┆ 6499.0 │
│ a96610ab360d42a2e5335a3998b471… ┆ 4799.0 │
│ 199af31afc78c699f0dbf71fb178d4… ┆ 4690.0 │
└──────────────────────────

In [42]:
def best_of(fn, n=20):
    times = []
    for _ in range(n):
        t0 = time.perf_counter()
        fn()
        times.append(time.perf_counter() - t0)
    return min(times) * 1000

print(f"sort + head: {best_of(lambda: items.sort('price', descending=True).head(10)):.2f} ms")
print(f"top_k:       {best_of(lambda: items.top_k(10, by='price')):.2f} ms")

sort + head: 13.88 ms
top_k:       5.64 ms


## Try it yourself

1. The 10 most recent orders per status, using `top_k_by` on `order_purchase_timestamp` inside `agg`.
2. Sort `items` by `seller_id`, then by `shipping_limit_date` newest first, with nulls last.
3. What does `reverse=True` do in `items.top_k(5, by="price", reverse=True)`? Compare with `bottom_k`.